# Step 10 — Is the LLM reproducible?

**Input** — `06_validation_set.csv`

**Output** — `llm_repeat_predictions.parquet` + `llm_repeat_comparison.csv`

Notebook 08 gives one answer per building. This one asks the model the **same

question `LLM_REPRO_RUNS` times** and puts the answers side by side.

That matters because every accuracy figure in notebook 09 is a single sample. If

the model returns a different class each time it is asked, a one-run score is a

draw from a distribution, not a measurement — and the gap between two arms can be

smaller than the model's own run-to-run spread.

The output sheet has one column per run, so disagreement is visible at a glance:

| gml_id | truth | run_1 | run_2 | run_3 | n_distinct | all_agree | modal_correct |

|---|---|---|---|---|---|---|---|

Set `SAMPLE_N` below to test on a subset first — at ~7 s/row, 3 runs over all 889

buildings is around 5 hours.

In [1]:
import sys

sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))

import time

from collections import Counter

import pandas as pd

from config import (VALIDATION_SET_FILE, LLM_REPRO_PREDICTIONS, LLM_REPRO_COMPARISON,

                    LLM_REPRO_RUNS, LLM_CHUNK_SIZE, LLM_MODEL, LLM_REASONING,

                    LLM_RETRY_SWEEPS, LLM_SWEEP_PAUSE_SEC)

from llm_utils import predict_row, normalise_mid_labels, SYSTEM_PROMPT

from validation_utils import (decode_final_validation_set, collapse_to_zone_activities,

                              resolve_prediction_bosserhof, format_label_set)

# None = every validated building. Set an integer to trial a subset first.

SAMPLE_N = None

SEED = 0

pd.set_option('display.width', 220)

print(f'model={LLM_MODEL}  reasoning={LLM_REASONING}  runs={LLM_REPRO_RUNS}  '

      f'prompt={len(SYSTEM_PROMPT):,} chars')

model=gpt-oss-120b  reasoning=high  runs=5  prompt=11,982 chars


In [2]:
val = decode_final_validation_set(pd.read_csv(VALIDATION_SET_FILE))

val['gml_id'] = val['gml_id'].astype(str)

if SAMPLE_N:

    val = val.sample(n=min(SAMPLE_N, len(val)), random_state=SEED).reset_index(drop=True)

print(f'{len(val):,} buildings x {LLM_REPRO_RUNS} runs = '

      f'{len(val) * LLM_REPRO_RUNS:,} calls (~{len(val) * LLM_REPRO_RUNS * 7 / 3600:.1f} h)')

889 buildings x 5 runs = 4,445 calls (~8.6 h)


## 1. Run the model N times

Checkpointed on `(gml_id, run)`, so an interrupted pass resumes without repeating

completed calls. The prompt is identical on every run — `call_tu_llm` is stateless

(`thread: None`), so the repeats cannot influence each other. Any difference

between runs is the model's own sampling, nothing else.

In [3]:
def append_checkpoint(path, rows):

    # keep='last' lets a successful retry overwrite an earlier failure.

    new = pd.DataFrame(rows)

    if path.exists():

        new = pd.concat([pd.read_parquet(path), new], ignore_index=True)

    new = new.drop_duplicates(['gml_id', 'run'], keep='last')

    path.parent.mkdir(parents=True, exist_ok=True)

    new.to_parquet(path, index=False)

def still_missing():

    """(gml_id, run) pairs with no SUCCESSFUL answer yet — errors count as missing.

    A failed call left in place would become a permanent hole in one run's column,

    and this notebook's whole output is a comparison ACROSS run columns: a hole

    would read as "the model changed its answer" when in fact it was never asked.

    """

    answered = set()

    if LLM_REPRO_PREDICTIONS.exists():

        prior = pd.read_parquet(LLM_REPRO_PREDICTIONS)

        ok = prior[prior['error'].isna()]

        answered = set(zip(ok['gml_id'].astype(str), ok['run']))

    # ORDER IS LOAD-BEARING — do not swap these two loops.

    #

    # `run` is the OUTER loop, so the whole dataset is classified once before any

    # building is asked a second time. Two repeats of the same prompt are therefore

    # separated by ~889 other calls and well over an hour.

    #

    # Row-major order (ask building A five times, then building B) would be the

    # obvious "optimisation" and would quietly invalidate the experiment. The

    # endpoint's caching behaviour is undocumented; if it memoises an identical

    # prompt at all, five back-to-back repeats could return one cached answer five

    # times and this notebook would report perfect reproducibility that is really

    # just a cache hit. Spacing the repeats means agreement has to be earned.

    return [(r.gml_id, r.sentence, run)

            for run in range(1, LLM_REPRO_RUNS + 1)

            for r in val.itertuples(index=False)

            if (str(r.gml_id), run) not in answered]

def run_calls(todo, label):

    t0 = time.time()

    for i in range(0, len(todo), LLM_CHUNK_SIZE):

        chunk = todo[i:i + LLM_CHUNK_SIZE]

        out = []

        for gid, sentence, run in chunk:

            r = predict_row(gid, sentence, fields='full', src_file=VALIDATION_SET_FILE.name)

            r['run'] = run

            out.append(r)

        append_checkpoint(LLM_REPRO_PREDICTIONS, out)

        n = i + len(chunk)

        rate = (time.time() - t0) / n

        print(f'{label} {n:,}/{len(todo):,} | errors {sum(1 for r in out if r["error"]):d} | '

              f'{rate:.1f} s/call | ETA {(len(todo) - n) * rate / 60:.1f} min')

todo = still_missing()

print(f'calls to make: {len(todo):,}')

if todo:

    run_calls(todo, 'pass 1')

for sweep in range(1, LLM_RETRY_SWEEPS + 1):

    failed = still_missing()

    if not failed:

        break

    pause = LLM_SWEEP_PAUSE_SEC * sweep

    print(f'\n{len(failed):,} call(s) still unanswered — sweep {sweep}/{LLM_RETRY_SWEEPS} '

          f'after a {pause}s pause')

    time.sleep(pause)

    run_calls(failed, f'sweep {sweep}')

outstanding = still_missing()

assert not outstanding, (

    f'{len(outstanding)} call(s) still unanswered after {LLM_RETRY_SWEEPS} sweeps: '

    f'{[(g, r) for g, _, r in outstanding[:10]]}. Re-run this cell — progress is '

    'checkpointed, so it costs only the failures. A hole in one run column would be '

    'indistinguishable from the model changing its answer.')

print(f'\nall {len(val) * LLM_REPRO_RUNS:,} calls answered.')

calls to make: 0

all 4,445 calls answered.


## 2. Build the side-by-side sheet

In [4]:
rep = pd.read_parquet(LLM_REPRO_PREDICTIONS)

rep['gml_id'] = rep['gml_id'].astype(str)

rep = rep[rep['gml_id'].isin(val['gml_id'])].drop_duplicates(['gml_id', 'run'], keep='last')

rep['boss'] = rep['bosserhof_class'].map(resolve_prediction_bosserhof)

rep['boss'] = rep['boss'].map(lambda v: '(no class)' if v in (None, '') else v)

rep['acts'] = rep['mid_labels'].map(normalise_mid_labels).map(

    collapse_to_zone_activities).map(format_label_set)

boss_wide = rep.pivot(index='gml_id', columns='run', values='boss').add_prefix('boss_run')

acts_wide = rep.pivot(index='gml_id', columns='run', values='acts').add_prefix('acts_run')

sheet = (val[['gml_id', 'osm_names', 'bosserhof_truth', 'activities_truth',

              'Bosserhof_class_mistakes_color', 'sentence']]

         .rename(columns={'Bosserhof_class_mistakes_color': 'boss_human_verdict'})

         .set_index('gml_id')

         .join(boss_wide).join(acts_wide))

sheet['bosserhof_truth'] = sheet['bosserhof_truth'].map(

    lambda v: '(no class)' if v == '' else v).fillna('')

sheet['activities_truth'] = sheet['activities_truth'].map(format_label_set)

boss_cols = list(boss_wide.columns)

acts_cols = list(acts_wide.columns)

def modal(values):
    # Most common answer across the runs; ties broken alphabetically.
    # Counter.most_common is insertion-ordered on a tie, so with an even
    # split (2-2-1 over five runs) it would return whichever answer happened
    # to be stored first, and re-running this cell on the same data could
    # report a different majority. Sorting the key makes the tie stable.
    vals = [v for v in values if pd.notna(v)]
    if not vals:
        return None
    counts = Counter(vals)
    return sorted(counts, key=lambda v: (-counts[v], v))[0]


sheet['boss_n_distinct'] = sheet[boss_cols].nunique(axis=1)

sheet['boss_all_agree']  = (sheet['boss_n_distinct'] == 1).astype(int)

sheet['boss_modal']      = sheet[boss_cols].apply(modal, axis=1)

sheet['boss_modal_correct'] = [

    '' if not t else int(str(m).strip().lower() == str(t).strip().lower())

    for m, t in zip(sheet['boss_modal'], sheet['bosserhof_truth'])]

sheet['acts_n_distinct'] = sheet[acts_cols].nunique(axis=1)

sheet['acts_all_agree']  = (sheet['acts_n_distinct'] == 1).astype(int)

ordered = (['osm_names', 'boss_human_verdict', 'bosserhof_truth'] + boss_cols +

           ['boss_n_distinct', 'boss_all_agree', 'boss_modal', 'boss_modal_correct',

            'activities_truth'] + acts_cols + ['acts_n_distinct', 'acts_all_agree', 'sentence'])

sheet = sheet.reset_index()[['gml_id'] + ordered]

sheet.to_csv(LLM_REPRO_COMPARISON, index=False, encoding='utf-8')

print(f'wrote {LLM_REPRO_COMPARISON.name}  ({len(sheet):,} rows)')

sheet.drop(columns='sentence').head(10)

wrote llm_repeat_comparison.csv  (889 rows)


,gml_id,osm_names,boss_human_verdict,bosserhof_truth,boss_run1,boss_run2,boss_run3,boss_run4,boss_run5,boss_n_distinct,...,boss_modal,boss_modal_correct,activities_truth,acts_run1,acts_run2,acts_run3,acts_run4,acts_run5,acts_n_distinct,acts_all_agree
0,236928,['Enoteca Vetrone'],green,restaurants gastronomy,restaurants gastronomy,restaurants gastronomy,restaurants gastronomy,restaurants gastronomy,restaurants gastronomy,1,...,restaurants gastronomy,1,Leisure; Workers,Leisure; Workers,Leisure; Workers,Leisure; Workers,Leisure; Workers,Leisure; Workers,1,1
1,388629,"[""Deutsche Bank;Ernsting's family""]",green,retail small scale,retail small scale,retail small scale,retail small scale,retail small scale,retail small scale,1,...,retail small scale,1,Retail_Non-Daily; Workers,Retail_Daily; Retail_Non-Daily; Workers,Retail_Daily; Retail_Non-Daily; Workers,Retail_Daily; Retail_Non-Daily; Workers,Retail_Daily; Retail_Non-Daily; Workers,Retail_Daily; Retail_Non-Daily; Workers,1,1
2,81391,NaN,green,industrial operations production,(no class),(no class),(no class),(no class),(no class),1,...,(no class),0,Workers,,,,,,1,1
3,36228,"['Nazar Trockenfrüchte', ""Sara's Collection"", ...",green,retail small scale,retail small scale,retail small scale,retail small scale,retail small scale,retail small scale,1,...,retail small scale,1,Retail_Daily; Retail_Non-Daily; University; Wo...,Leisure; Retail_Daily; Retail_Non-Daily; Unive...,Leisure; Retail_Daily; Retail_Non-Daily; Unive...,Leisure; Retail_Daily; Retail_Non-Daily; Unive...,Leisure; Retail_Daily; Retail_Non-Daily; Unive...,Leisure; Retail_Daily; Retail_Non-Daily; Unive...,1,1
4,556581,['Reni'],green,hotels,hotels,hotels,hotels,hotels,hotels,1,...,hotels,1,Leisure; Workers,Leisure; Workers,Leisure; Workers,Leisure; Workers,Leisure; Workers,Leisure; Workers,1,1
5,389444,NaN,green,customer oriented services,customer oriented services,normal office,normal office,(no class),normal office,3,...,normal office,0,Retail_Daily; Retail_Non-Daily; Workers,Retail_Daily; Workers,Retail_Daily; Workers,Retail_Daily; Workers,,Retail_Daily; Workers,2,0
6,453160,NaN,red,business oriented services,(no class),(no class),(no class),customer oriented services,(no class),2,...,(no class),0,Retail_Non-Daily,,Workers,,Retail_Daily,,3,0
7,21469,[Kunstatelier],red,entertainment culture,normal office,services,(no class),normal office,normal office,3,...,normal office,0,Leisure; Workers,Workers,Workers,,Workers,Workers,2,0
8,90811,['Biergarten des griechischen Restaurants'],green,restaurants gastronomy,(no class),(no class),(no class),(no class),(no class),1,...,(no class),0,Leisure; Workers,,,,,,1,1
9,175639,NaN,red,retail small scale,services,services,(no class),retail,(no class),3,...,(no class),0,Retail_Non-Daily; Workers,,,,Retail_Daily; Retail_Non-Daily; Workers,,2,0


## 3. How reproducible is it?

`all_agree` is the headline: the share of buildings where every run returned the

same answer. Anything below 100% means a single-run accuracy figure carries an

error bar the score itself does not show.

In [5]:
n = len(sheet)

print(f'buildings: {n:,}   runs each: {LLM_REPRO_RUNS}\n')

print('BOSSERHOF')

print(f"  identical across all runs : {sheet['boss_all_agree'].sum():,} / {n:,} "

      f"({sheet['boss_all_agree'].mean():.1%})")

print(f"  distinct answers per building: {sheet['boss_n_distinct'].mean():.2f} avg, "

      f"{sheet['boss_n_distinct'].max()} max")

print(sheet['boss_n_distinct'].value_counts().sort_index().to_string())

print('\nACTIVITIES')

print(f"  identical across all runs : {sheet['acts_all_agree'].sum():,} / {n:,} "

      f"({sheet['acts_all_agree'].mean():.1%})")

scoreable = sheet[sheet['boss_modal_correct'] != '']

if len(scoreable):

    print(f"\nmajority-vote Bosserhof accuracy: "

          f"{scoreable['boss_modal_correct'].astype(int).mean():.1%} "

          f"({scoreable['boss_modal_correct'].astype(int).sum():,}/{len(scoreable):,})")

per_run = []

for c in boss_cols:

    ok = [int(str(p).strip().lower() == str(t).strip().lower())

          for p, t in zip(sheet[c], sheet['bosserhof_truth']) if t]

    per_run.append({'run': c, 'accuracy': round(sum(ok) / len(ok), 4) if ok else None,

                    'correct': sum(ok), 'of': len(ok)})

print('\nper-run Bosserhof accuracy (the spread a single run hides):')

print(pd.DataFrame(per_run).to_string(index=False))

buildings: 889   runs each: 5

BOSSERHOF
  identical across all runs : 550 / 889 (61.9%)
  distinct answers per building: 1.52 avg, 5 max
boss_n_distinct
1    550
2    225
3    103
4      9
5      2

ACTIVITIES
  identical across all runs : 457 / 889 (51.4%)

majority-vote Bosserhof accuracy: 70.6% (619/877)

per-run Bosserhof accuracy (the spread a single run hides):
      run  accuracy  correct  of
boss_run1    0.6887      604 877
boss_run2    0.6784      595 877
boss_run3    0.6899      605 877
boss_run4    0.6876      603 877
boss_run5    0.6887      604 877


## 4. What actually changes between runs

If the flips were confined to neighbouring subcategories they would be a

granularity artefact. Crossing headline categories means the model is

reclassifying the building's *type*, which is a different and more serious kind of

instability.

In [6]:
flips = sheet[sheet['boss_all_agree'] == 0]

print(f'{len(flips):,} buildings changed answer between runs\n')

if len(flips):

    pairs = Counter()

    for _, r in flips.iterrows():

        vals = sorted({v for v in r[boss_cols] if pd.notna(v)})

        for i in range(len(vals)):

            for j in range(i + 1, len(vals)):

                pairs[(vals[i], vals[j])] += 1

    top = pd.DataFrame([{'class_a': a, 'class_b': b, 'n': n_} for (a, b), n_ in pairs.most_common(20)])

    print(top.to_string(index=False))

339 buildings changed answer between runs



                                                         class_a                                   class_b  n
                                      customer oriented services                        retail small scale 44
                                      customer oriented services                             normal office 35
                                      customer oriented services                                  services 32
                                           entertainment culture                         public facilities 24
                                                   normal office                                  services 21
                                                      (no class)                customer oriented services 20
                                                      (no class)                                  services 20
                                              retail small scale                                  services 20
highly pro